<a href="https://colab.research.google.com/github/solosolve-ai/solosolve-ai/blob/main/manim_video_creator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#installs

In [1]:
# Cell 1: Installation of Manim, ManimML, and dependencies
print("Starting installation... This will take 5-10 minutes.")
!sudo apt-get update -y
!sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg texlive-latex-base texlive-fonts-recommended texlive-fonts-extra texlive-latex-extra
!pip install --upgrade pip setuptools wheel
!pip install manim==0.19.0 manimpango==0.5.0 manim-ml
print("Installation Complete!")

Starting installation... This will take 5-10 minutes.
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to pr

#video

In [2]:
# Cell 2: Chapter 2 Scene Definitions (with ManimML)

from manim import *
from scipy.stats import norm
import numpy as np
# Import ManimML
from manim_ml.neural_network import NeuralNetwork, FeedForwardLayer

# --- SCENE 2.1: The Soul of the Transformer (Attention) ---

class AttentionManifoldScene(ThreeDScene):
    def construct(self):
        title = Text("The Soul of the Transformer: Self-Attention", font_size=36).to_corner(UL).set_z_index(10)
        formula = MathTex(r"\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V", font_size=40).to_corner(UR).set_z_index(10)
        self.add_fixed_in_frame_mobjects(title, formula)
        self.set_camera_orientation(phi=75 * DEGREES, theta=30 * DEGREES, zoom=0.8)

        # FIX: Use Surface, which is the correct class name
        manifold = Surface(
            lambda u, v: np.array([u, v, 0.2 * np.sin(u) * np.cos(v)]),
            u_range=[-5, 5], v_range=[-5, 5],
            checkerboard_colors=[BLUE_D, BLUE_E], resolution=(15, 15)
        ).scale(1.5)

        self.play(Write(title), Write(formula))
        self.play(Create(manifold))
        self.begin_ambient_camera_rotation(rate=0.1)

        tokens = {
            "shirt": Dot3D(point=np.array([-2, 1, 0.1]), color=YELLOW),
            "blue": Dot3D(point=np.array([0.5, -2, 0.2]), color=WHITE),
            "torn": Dot3D(point=np.array([-3, -1.5, -0.3]), color=WHITE)
        }
        token_labels = VGroup(*[Text(t, font_size=24).next_to(d, OUT) for t, d in tokens.items()])
        self.add_fixed_in_frame_mobjects(*token_labels)
        self.play(LaggedStart(*[Create(d) for d in tokens.values()], lag_ratio=0.5))

        q_vec = Arrow3D(start=tokens["shirt"].get_center(), end=tokens["shirt"].get_center() + np.array([1, 1, 1]), color=BLUE)
        k_vecs = VGroup(*[Arrow3D(start=d.get_center(), end=d.get_center() + np.array([-1, 0.5, 0.5]), color=RED) for d in tokens.values()])
        v_vecs = VGroup(*[Arrow3D(start=d.get_center(), end=d.get_center() + np.array([0, -1, 1]), color=GREEN) for d in tokens.values()])
        self.play(GrowArrow(q_vec), Create(k_vecs), Create(v_vecs))
        self.wait(1)

        scores_rect = SurroundingRectangle(formula.get_part_by_tex("QK^T"), color=YELLOW)
        self.play(Create(scores_rect))
        self.play(ShowPassingFlash(q_vec.copy().set_color(BLUE), time_width=0.5, run_time=2))
        self.play(FadeOut(scores_rect))

        softmax_rect = SurroundingRectangle(formula.get_part_by_tex("softmax"), color=YELLOW)
        self.play(Create(softmax_rect))
        glow_blue = Circle(radius=0.5, color=BLUE).move_to(tokens["blue"])
        glow_torn = Circle(radius=0.8, color=BLUE).move_to(tokens["torn"])
        self.play(FadeIn(glow_blue, glow_torn, scale=2))

        weighted_v_blue = v_vecs[1].copy().scale(0.5, about_point=v_vecs[1].get_start())
        weighted_v_torn = v_vecs[2].copy().scale(0.8, about_point=v_vecs[2].get_start())
        final_vec_target = tokens["shirt"].get_center() + weighted_v_blue.get_vector() + weighted_v_torn.get_vector()

        self.play(
            FadeOut(q_vec, k_vecs, v_vecs, glow_blue, glow_torn, softmax_rect),
            tokens["shirt"].animate.move_to(final_vec_target), run_time=2
        )
        self.wait(2)
        self.stop_ambient_camera_rotation()


# --- SCENE 2.2: Gemma's Architectural Efficiencies (with ManimML) ---

class GemmaEfficiencyScene(Scene):
    def construct(self):
        # --- Part 1: Grouped-Query Attention (GQA) ---
        title = Text("Gemma Efficiency 1: Grouped-Query Attention", font_size=36).to_edge(UP)
        self.play(Write(title))

        queries = VGroup(*[Circle(radius=0.2, color=BLUE) for _ in range(8)]).arrange(RIGHT, buff=0.3).shift(UP*1.5)
        query_text = Text("Queries (Q)", font_size=28).next_to(queries, LEFT)
        keys = VGroup(*[Square(side_length=0.4, color=RED) for _ in range(2)]).arrange(RIGHT, buff=2.2).shift(DOWN*1)
        values = VGroup(*[Triangle(color=GREEN).scale(0.25) for _ in range(2)]).arrange(RIGHT, buff=2).next_to(keys, DOWN, buff=0.2)
        kv_text = Text("Shared K/V", font_size=28).next_to(VGroup(keys, values), LEFT)

        self.play(Write(query_text), Create(queries))
        self.play(Write(kv_text), Create(keys), Create(values))

        group_boxes = VGroup(SurroundingRectangle(queries[0:4]), SurroundingRectangle(queries[4:8]))
        lines = VGroup(*[Arrow(queries[i].get_bottom(), keys[i // 4].get_top(), buff=0.1) for i in range(8)])
        self.play(Create(group_boxes), Create(lines))
        self.wait(2)
        self.play(FadeOut(queries, query_text, keys, values, kv_text, group_boxes, lines))

        # --- Part 2: Interleaved Attention (with ManimML) ---
        new_title = Text("Gemma Efficiency 2: Interleaved Attention", font_size=36).to_edge(UP)
        self.play(Transform(title, new_title))

        # Define layers for ManimML
        layers = []
        for i in range(12):
            if (i + 1) % 6 == 0: # Global layer
                layers.append(FeedForwardLayer(num_nodes=8, node_color=YELLOW))
            else: # Local layer
                layers.append(FeedForwardLayer(num_nodes=4, node_color=BLUE))

        nn = NeuralNetwork(layers, layer_spacing=0.4).scale(0.8).center()
        self.play(Create(nn))

        labels = VGroup(
            Text("Local Layer", font_size=24).next_to(nn.layers[0], RIGHT),
            Text("Global Layer", font_size=24).next_to(nn.layers[5], RIGHT)
        )
        self.play(Write(labels))

        # Animate forward pass
        self.play(nn.make_forward_pass_animation(run_time=4))
        self.wait(2)


# --- SCENE 2.3: The Surgical Tools (QLoRA) ---

class QLoRASurgeryScene(Scene):
    def construct(self):
        title = Text("QLoRA Part 1: Quantization (NF4)", font_size=36).to_edge(UP)
        self.play(Write(title))

        axes = Axes(x_range=[-4, 4, 1], y_range=[0, 0.5, 0.1], x_length=8, y_length=4)
        curve = axes.plot(lambda x: norm.pdf(x), x_range=[-4, 4], color=BLUE)
        self.play(Create(axes), Create(curve))

        num_quantiles = 16
        quantiles = norm.ppf(np.linspace(0.01, 0.99, num_quantiles + 1))

        # FIX: Use x_range instead of x_bounding_points
        rects = axes.get_riemann_rectangles(curve, x_range=[quantiles[0], quantiles[-1]], dx=0.2, color=[BLUE, GREEN])

        self.play(Create(rects))
        explanation = Text("Values are chosen based on equal probability areas", font_size=24).to_edge(DOWN)
        self.play(Write(explanation))
        self.wait(2)
        self.play(FadeOut(axes, curve, rects, explanation))

        new_title = Text("QLoRA Part 2: Low-Rank Adaptation", font_size=36).to_edge(UP)
        self.play(Transform(title, new_title))

        formula = MathTex(r"y = ", r"\mathbf{W}_0 x", r" + \frac{\alpha}{r}", r"\mathbf{B}", r"\mathbf{A}", r"x").scale(1.2)
        self.play(Write(formula))

        w0_box = SurroundingRectangle(formula.get_part_by_tex("W_0"), color=GRAY)
        w0_text = Text("Frozen & Quantized", font_size=24, color=GRAY).next_to(w0_box, DOWN)
        lora_box = SurroundingRectangle(VGroup(formula.get_part_by_tex("B"), formula.get_part_by_tex("A")), color=YELLOW)
        lora_text = Text("Trainable Adapters", font_size=24, color=YELLOW).next_to(lora_box, DOWN)

        self.play(Create(w0_box), Write(w0_text))
        self.play(Create(lora_box), Write(lora_text))
        self.wait(3)


# --- SCENE 2.4: The Full Training Cycle (with ManimML) ---

class TrainingCycleScene(Scene):
    def construct(self):
        title = Text("The Training Cycle", font_size=36).to_edge(UP)
        self.play(Write(title))

        # --- Setup the visual pipeline with ManimML ---
        nn = NeuralNetwork([
            FeedForwardLayer(num_nodes=3, layer_title="Input"),
            FeedForwardLayer(num_nodes=5, layer_title="Gemma + LoRA"),
            FeedForwardLayer(num_nodes=3, layer_title="Heads"),
            FeedForwardLayer(num_nodes=4, layer_title="Logits")
        ], layer_spacing=1.5).scale(0.9)
        self.play(Create(nn))

        # --- 1. Forward Pass ---
        self.play(nn.make_forward_pass_animation(run_time=3))
        self.wait(0.5)

        # --- 2. Loss Calculation ---
        gold_labels = Text("Gold Labels", font_size=24).next_to(nn.layers[-1], DOWN, buff=1.5)
        loss_formula = MathTex(r"L = \text{Loss}(\text{Logits}, \text{Labels})", color=RED).next_to(gold_labels, RIGHT, buff=1)

        self.play(Write(gold_labels))
        self.play(
            LaggedStart(
                Arrow(nn.layers[-1].get_bottom(), loss_formula.get_left()),
                Arrow(gold_labels.get_right(), loss_formula.get_left()),
                lag_ratio=0.3
            ),
            Write(loss_formula)
        )
        self.wait(1)

        # --- 3. Backpropagation ---
        backprop_title = Text("Backpropagation", font_size=28).to_edge(DOWN)
        self.play(Write(backprop_title))

        # ManimML's built-in backprop is perfect for this
        self.play(nn.make_backward_pass_animation(run_time=3))

        lora_layer = nn.layers[1]
        lora_highlight = SurroundingRectangle(lora_layer, color=YELLOW)
        update_text = Text("Only LoRA adapters are updated", font_size=24, color=YELLOW).next_to(lora_highlight, DOWN)
        self.play(Create(lora_highlight), Write(update_text))
        self.wait(2)

        # --- 4. Optimizer Step ---
        self.play(FadeOut(backprop_title, lora_highlight, update_text))

        optimizer_formula = MathTex(r"\theta_{new} = \theta_{old} - \eta \nabla L").to_edge(DOWN)
        self.play(Write(optimizer_formula))

        # FIX: Replace SVG with a robust MathTex Mobject
        optimizer_symbol = MathTex(r"\odot", font_size=96, color=YELLOW).move_to(lora_layer.get_center())
        self.play(FadeIn(optimizer_symbol, scale=0.5))
        self.play(Rotate(optimizer_symbol, angle=PI*2, rate_func=linear, run_time=2))
        self.wait(2)

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [3]:
# Cell 3: Render the Attention Scene (3D)
%%manim -pql AttentionManifoldScene

Manim Community v0.19.0

[11/02/25 12:07:16] INFO     Writing \text{Attention}(Q, K, V) =                            ]8;id=468627;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py\tex_file_writing.py]8;;\:]8;id=37577;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py#111\111]8;;\
                             \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V to                                
                             media/Tex/d5a45b65da6ba0c4.tex                                                        

[11/02/25 12:07:22] INFO     Animation 0 : Partial movie file written in                   ]8;id=14638;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=396782;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Att                         
                             entionManifoldScene/2975571396_2264112269_1651289494.mp4'                             

[11/02/25 12:07:31] INFO     Animation 1 : Partial movie file written in                   ]8;id=778098;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=932377;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Att                         
                             entionManifoldScene/639311802_2100890907_90575924.mp4'                                

[11/02/25 12:07:43] INFO     Animation 2 : Partial movie file written in                   ]8;id=534730;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=593087;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Att                         
                             entionManifoldScene/806262440_280220432_10794572.mp4'                                 

TypeError: Mobject.apply_points_function_about_point() got an unexpected keyword argument 'scale_tips'

In [4]:
# Cell 4: Render the Gemma Efficiency Scene
%%manim -pql GemmaEfficiencyScene

Manim Community v0.19.0

[11/02/25 12:22:01] INFO     Animation 0 : Partial movie file written in                   ]8;id=290407;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=35324;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Gem                         
                             maEfficiencyScene/1185818338_3733031669_223132457.mp4'                                

[11/02/25 12:22:02] INFO     Animation 1 : Partial movie file written in                   ]8;id=14090;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=58558;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Gem                         
                             maEfficiencyScene/624642324_3410062998_1874569469.mp4'                                

                    INFO     Animation 2 : Partial movie file written in                   ]8;id=57148;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=360507;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Gem                         
                             maEfficiencyScene/624642324_4008795436_777432167.mp4'                                 

[11/02/25 12:22:03] INFO     Animation 3 : Partial movie file written in                   ]8;id=527284;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=366901;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Gem                         
                             maEfficiencyScene/624642324_1676599906_3552884101.mp4'                                

                    INFO     Animation 4 : Partial movie file written in                   ]8;id=162230;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=748815;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Gem                         
                             maEfficiencyScene/624642324_2872842549_3740669622.mp4'                                

[11/02/25 12:22:04] INFO     Animation 5 : Partial movie file written in                   ]8;id=145178;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=722630;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Gem                         
                             maEfficiencyScene/624642324_4236123317_279524822.mp4'                                 

[11/02/25 12:22:05] INFO     Animation 6 : Partial movie file written in                   ]8;id=967895;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=444567;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Gem                         
                             maEfficiencyScene/624642324_4194093274_622328241.mp4'                                 

Constructing layers
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
NeuralNetwork([
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(input_layer=FeedForwardLayer,output_layer=FeedForwardLayer,)(z_index=2, title_text= , ),
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(input_layer=FeedForwardLayer,output_layer=FeedForwardLayer,)(z_index=2, title_text= , ),
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(input_layer=FeedForwardLayer,output_layer=FeedForwardLayer,)(z_index=2, title_text= , ),
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(i

AttributeError: 'NoneType' object has no attribute 'center'

In [5]:
# Cell 5: Render the QLoRA Scene
%%manim -pql QLoRASurgeryScene

Manim Community v0.19.0

[11/02/25 12:23:55] INFO     Animation 0 : Partial movie file written in                   ]8;id=570453;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=39660;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/QLo                         
                             RASurgeryScene/1185818338_4265206383_223132457.mp4'                                   

                    INFO     Animation 1 : Partial movie file written in                   ]8;id=789759;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=978206;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/QLo                         
                             RASurgeryScene/624642324_2499798980_144750668.mp4'                                    

[11/02/25 12:23:56] INFO     Animation 2 : Partial movie file written in                   ]8;id=926469;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=514748;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/QLo                         
                             RASurgeryScene/624642324_3997894729_2697946714.mp4'                                   

[11/02/25 12:23:57] INFO     Animation 3 : Partial movie file written in                   ]8;id=469645;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=674753;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/QLo                         
                             RASurgeryScene/624642324_2768597617_772964833.mp4'                                    

...

                    INFO     Writing \mathbf{B} to media/Tex/f74fb75d52ed98aa.tex           ]8;id=608351;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py\tex_file_writing.py]8;;\:]8;id=180187;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py#111\111]8;;\

[11/02/25 12:24:03] INFO     Writing \mathbf{A} to media/Tex/a50db743398cc1b3.tex           ]8;id=58956;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py\tex_file_writing.py]8;;\:]8;id=484595;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py#111\111]8;;\

                    INFO     Writing x to media/Tex/d3c1af651a272204.tex                    ]8;id=613161;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py\tex_file_writing.py]8;;\:]8;id=264265;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py#111\111]8;;\

[11/02/25 12:24:04] INFO     Animation 7 : Partial movie file written in                   ]8;id=547491;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=546472;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/QLo                         
                             RASurgeryScene/624642324_2791808266_1866453155.mp4'                                   

TypeError: Expected all inputs for parameter mobjects to be a Mobjects

In [6]:
# Cell 6: Render the Training Cycle Scene
%%manim -pql TrainingCycleScene

Manim Community v0.19.0

[11/02/25 12:24:15] INFO     Animation 0 : Partial movie file written in                   ]8;id=167499;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=274778;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Tra                         
                             iningCycleScene/1185818338_1313241714_223132457.mp4'                                  

Constructing layers
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
NeuralNetwork([
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(input_layer=FeedForwardLayer,output_layer=FeedForwardLayer,)(z_index=2, title_text= , ),
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(input_layer=FeedForwardLayer,output_layer=FeedForwardLayer,)(z_index=2, title_text= , ),
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(input_layer=FeedForwardLayer,output_layer=FeedForwardLayer,)(z_index=2, title_text= , ),
    FeedForwardLayer(z_index=3, title_text= , ),
])


NotImplementedError: This animation is not defined for this Mobject.